# 📈 Crypto Trading Firm Interview Question Bank
### Buy-Side & Sell-Side | Math • Scenario • Terminology
---
> **How to use this notebook:** Each section contains interview questions with detailed answers. Run the code cells for quantitative/math-based worked examples. Use this as a study guide or interviewer reference.

## Table of Contents
1. [Key Terminology](#terminology)
2. [Math & Calculation Questions](#math)
3. [Market Microstructure & Order Types](#microstructure)
4. [Crypto-Specific Concepts](#crypto)
5. [Risk Management Questions](#risk)
6. [Scenario-Based Questions](#scenario)
7. [Derivatives & Options](#derivatives)
8. [DeFi & On-Chain Trading](#defi)
9. [Behavioral / Fit Questions](#behavioral)

In [ ]:
# Setup: Install/import required libraries for worked examples
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded. Ready to work through crypto trading interview questions!")

---
<a id='terminology'></a>
## 1. Key Terminology
> Questions testing your fluency with day-to-day crypto trading language.

### Q1.1 — What is the bid-ask spread and why does it matter in crypto markets?

**Answer:**  
The **bid-ask spread** is the difference between the highest price a buyer is willing to pay (bid) and the lowest price a seller is willing to accept (ask).  

- **Bid** = what market makers will *buy* from you  
- **Ask** = what market makers will *sell* to you  
- **Spread** = Ask − Bid

**Why it matters in crypto:**  
- Spreads are wider on illiquid pairs (e.g. SHIB/USDT on a minor DEX) vs tight on BTC/USDT on Binance  
- The spread is a **transaction cost** for takers  
- Market makers earn the spread as compensation for providing liquidity  
- During high volatility (e.g. a Flash Crash or major news), spreads widen dramatically — a 0.01% spread on BTC can jump to 0.5%+

### Q1.2 — Explain the difference between a Market Maker and a Market Taker.

**Answer:**  
| | Market Maker | Market Taker |
|---|---|---|
| **Action** | Posts limit orders on the book | Executes against existing orders |
| **Liquidity** | Adds liquidity | Removes liquidity |
| **Fee** | Earns a rebate (or pays lower fee) | Pays higher fee |
| **Crypto example** | A firm quoting BTC/USDT bid/ask on Binance 24/7 | A retail trader placing a market buy of BTC |

**Key insight:** On most crypto exchanges (Binance, Coinbase, Bybit), makers pay ~0 to 0.02% and takers pay ~0.04 to 0.10%. This fee structure incentivises firms to provide continuous two-sided quotes.

### Q1.3 — What is slippage? How does it differ in spot vs perpetual futures?

**Answer:**  
**Slippage** is the difference between the expected execution price and the actual fill price, caused by insufficient liquidity at a given price level.

- **Spot BTC:** Buy 100 BTC on Binance — the order book may only have 20 BTC at $65,000, then 30 at $65,050, then 50 at $65,150. Your average fill is above $65,000.
- **BTC Perpetual Futures:** Same concept, but the funding rate mechanism and index price anchor reduce dislocation vs spot. However, during liquidation cascades, perp slippage can be more severe due to forced liquidations consuming book depth.

**Formula:** `Slippage % = (Fill Price − Expected Price) / Expected Price × 100`

### Q1.4 — Define: VWAP, TWAP, and how they are used in crypto execution.

**Answer:**  
**VWAP (Volume-Weighted Average Price):**  
The average price weighted by volume traded. Used as an execution benchmark — a good execution is below VWAP (for buys).  
`VWAP = Σ(Price × Volume) / Σ(Volume)`

**TWAP (Time-Weighted Average Price):**  
Breaks a large order into equal slices over a time period, regardless of volume. More predictable; used when minimising market impact is key.  
`TWAP = Σ(Price at each interval) / Number of intervals`

**Crypto context:**  
- A fund buying $50M of ETH may use a TWAP algo over 4 hours to avoid moving the market  
- VWAP is commonly used by OTC desks to price large block trades  
- In 24/7 crypto markets (no open/close), these metrics are computed over rolling windows

### Q1.5 — What is a funding rate in perpetual futures and what does it signal?

**Answer:**  
**Funding rate** is a periodic payment between long and short holders in perpetual futures contracts, designed to keep the contract price anchored to the spot price.

- **Positive funding rate:** Longs pay shorts → market is bullish/over-leveraged to the long side  
- **Negative funding rate:** Shorts pay longs → market is bearish/over-leveraged to the short side  
- Paid every 8 hours on most exchanges (Binance, Bybit)

**Signal interpretation:**  
- Extremely high positive funding (e.g. +0.1% per 8hr = +109% annualised) signals euphoric leverage — a contrarian short signal  
- Deep negative funding signals fear/short squeeze potential  
- Trading firms monitor funding rate as a carry signal — going short perp + long spot = funding capture with delta-neutral exposure

### Q1.6 — What is the difference between Centralised Exchange (CEX) and Decentralised Exchange (DEX)?

**Answer:**  

| Feature | CEX (e.g. Binance, Coinbase) | DEX (e.g. Uniswap, dYdX) |
|---|---|---|
| **Custody** | Exchange holds your assets | You hold your keys |
| **Order matching** | Central limit order book (CLOB) | Automated Market Maker (AMM) or on-chain CLOB |
| **KYC** | Required | Usually not required |
| **Speed** | Millisecond matching | Block time latency (seconds) |
| **Counterparty risk** | Exchange default risk (FTX collapse 2022) | Smart contract risk |
| **Liquidity** | Generally deeper | Fragmented across pools |

**Trading firm implication:** Most HFT/market making occurs on CEXs due to speed. DEX arbitrage is a growing area — firms exploit price dislocations between Uniswap pools and Binance spot.

### Q1.7 — Define: Open Interest (OI), Long/Short Ratio, and Liquidation Cascade.

**Answer:**  
**Open Interest (OI):** Total number of outstanding derivative contracts (futures/options) that have not been settled. Rising OI with rising price = bullish trend with conviction. Rising OI with falling price = bearish trend building.

**Long/Short Ratio:** Ratio of users holding net long vs net short positions. A ratio > 1 means more traders are long. Extreme readings (e.g. 4:1 long) can precede sharp drops as longs get liquidated.

**Liquidation Cascade:** When leveraged positions are force-closed (liquidated) by the exchange because margin falls below maintenance margin. This triggers further selling/buying which liquidates adjacent positions — a feedback loop. Famous example: May 2021 BTC crash from $58k to $30k involved $8B+ in liquidations within 24 hours.

### Q1.8 — What is basis in crypto futures, and how do traders exploit it?

**Answer:**  
**Basis = Futures Price − Spot Price**

- **Positive basis (contango):** Futures trade at a premium to spot — common in crypto bull markets. Reflects cost of carry and bullish sentiment.
- **Negative basis (backwardation):** Futures trade at a discount — rare in crypto, signals strong bearish sentiment.

**Basis trade (cash-and-carry arbitrage):**  
1. Buy BTC spot  
2. Sell BTC futures at a premium  
3. Hold until futures expiry — collect the basis as profit  
4. Risk-free if basis > financing cost  

Example: If BTC spot = $65,000 and Dec futures = $66,000, the basis is $1,000 (1.54%). A fund can lock in ~6% annualised return with low directional risk.

---
<a id='math'></a>
## 2. Math & Calculation Questions
> Quantitative problems you may face on the spot in interviews.

**Notional Position Value**

$$ \text{Notional Value} = \text{Entry Price} \times \text{BTC Quantity} $$

$$ \text{Notional Value} = P_{\text{entry}} \times Q $$

---

**Margin Required (Initial Margin)**

$$ \text{Margin Required} = \frac{\text{Notional Value}}{\text{Leverage}} $$

$$ M = \frac{P_{\text{entry}} \times Q}{L} $$

---

**Gross P&L**

$$ \text{P\&L} = (\text{Exit Price} - \text{Entry Price}) \times \text{BTC Quantity} $$

$$ \text{P\&L} = (P_{\text{exit}} - P_{\text{entry}}) \times Q $$

---

**Return on Margin (RoM)**

$$ \text{Return on Margin} = \left( \frac{\text{P\&L}}{\text{Margin Required}} \right) \times 100 $$

$$ \text{RoM} = \left( \frac{(P_{\text{exit}} - P_{\text{entry}}) \times Q}{M} \right) \times 100\% $$

---

**Return on Notional**

$$ \text{Return on Notional} = \left( \frac{\text{Exit Price} - \text{Entry Price}}{\text{Entry Price}} \right) \times 100 $$

$$ \text{RoN} = \left( \frac{P_{\text{exit}} - P_{\text{entry}}}{P_{\text{entry}}} \right) \times 100\% $$

### Q2.1 — Calculate the P&L of a BTC futures trade.

**Question:** You go long 10 BTC at $62,000 using 5x leverage. BTC rises to $65,000. What is your P&L and return on margin?


In [ ]:
# Q2.1 — BTC Futures P&L Calculation

entry_price = 62_000       # USD per BTC
exit_price  = 65_000       # USD per BTC
btc_quantity = 10          # BTC
leverage = 5               # 5x leverage

# Notional position value
notional_value = entry_price * btc_quantity

# Margin required (initial margin = notional / leverage)
margin_required = notional_value / leverage

# Gross P&L
pnl = (exit_price - entry_price) * btc_quantity

# Return on margin
return_on_margin = (pnl / margin_required) * 100

# Return on notional
return_on_notional = ((exit_price - entry_price) / entry_price) * 100

print("=" * 50)
print("BTC FUTURES TRADE — P&L SUMMARY")
print("=" * 50)
print(f"Entry Price:          ${entry_price:,.0f}")
print(f"Exit Price:           ${exit_price:,.0f}")
print(f"Quantity:             {btc_quantity} BTC")
print(f"Leverage:             {leverage}x")
print(f"Notional Value:       ${notional_value:,.0f}")
print(f"Margin Required:      ${margin_required:,.0f}")
print("-" * 50)
print(f"Gross P&L:            ${pnl:,.0f}")
print(f"Return on Notional:   {return_on_notional:.2f}%")
print(f"Return on Margin:     {return_on_margin:.2f}%  ← Leverage amplification")
print("=" * 50)

# Liquidation price calculation
# For a long: Liquidation when loss = margin → entry - (margin/qty)
maintenance_margin_rate = 0.005  # 0.5% maintenance margin (typical on Binance)
liq_price = entry_price * (1 - (1/leverage) + maintenance_margin_rate)
print(f"\nApprox. Liquidation Price: ${liq_price:,.0f}")
print(f"  (a {((entry_price - liq_price)/entry_price)*100:.1f}% move against position triggers forced close)")

### Q2.2 — Calculate the funding rate cost.

**Question:** You hold a $500,000 long position in BTC perpetual futures. The funding rate is +0.03% every 8 hours. How much do you pay daily and monthly?

**When you are Long with Positive Funding Rate**  
→ You **pay** the shorts

---

**Cost per Funding Period**

$$ \text{Cost per Period} = \text{Position Size (USD)} \times \text{Funding Rate} $$

$$ C_{\text{period}} = \text{Notional} \times r_f $$

---

**Cost per Day**

$$ \text{Cost per Day} = \text{Cost per Period} \times \text{Periods per Day} $$

$$ C_{\text{day}} = C_{\text{period}} \times n_{\text{daily}} $$

---

**Cost per Week**

$$ \text{Cost per Week} = \text{Cost per Day} \times 7 $$

$$ C_{\text{week}} = C_{\text{day}} \times 7 $$

---

**Cost per Month (approx.)**

$$ \text{Cost per Month} = \text{Cost per Day} \times 30 $$

$$ C_{\text{month}} = C_{\text{day}} \times 30 $$

---

**Annualised Funding Rate (%)**

$$ \text{Annualised Rate} = r_f \times n_{\text{daily}} \times 365 \times 100 $$

$$ \text{Annualised (\%)} = r_f \times \text{Periods per Day} \times 365 \times 100 $$

In [ ]:
# Q2.2 — Funding Rate Cost Calculation

position_size_usd = 500_000   # $500k notional long
funding_rate = 0.0003         # 0.03% per 8-hour period
periods_per_day = 3           # 24hrs / 8hrs = 3 periods

# As a long with positive funding, you PAY shorts
cost_per_period = position_size_usd * funding_rate
cost_per_day    = cost_per_period * periods_per_day
cost_per_week   = cost_per_day * 7
cost_per_month  = cost_per_day * 30
annualised_rate = funding_rate * periods_per_day * 365 * 100  # in %

print("=" * 50)
print("FUNDING RATE COST — LONG POSITION")
print("=" * 50)
print(f"Position Size:        ${position_size_usd:,.0f}")
print(f"Funding Rate:         {funding_rate*100:.3f}% per 8 hours")
print("-" * 50)
print(f"Cost per period:      ${cost_per_period:,.2f}")
print(f"Cost per day:         ${cost_per_day:,.2f}")
print(f"Cost per week:        ${cost_per_week:,.2f}")
print(f"Cost per month:       ${cost_per_month:,.2f}")
print(f"Annualised rate:      {annualised_rate:.2f}%")
print("=" * 50)
print("\n💡 Interview insight: If you earn alpha < annualised funding cost,")
print("   your strategy is unprofitable on a carry-adjusted basis.")

# Bonus: Break-even daily move to justify holding
breakeven_daily_move = (cost_per_day / position_size_usd) * 100
print(f"\nBreak-even daily move needed: {breakeven_daily_move:.4f}%")

### Q2.3 — VWAP Calculation

**Question:** Given the following BTC trades, calculate the VWAP.

In [ ]:
# Q2.3 — VWAP Calculation

# BTC trades: (price, volume in BTC)
trades = [
    (64_800, 2.5),
    (64_850, 1.2),
    (64_900, 4.0),
    (64_950, 0.8),
    (65_000, 3.3),
    (65_050, 2.1),
    (65_100, 5.5),
]

prices  = np.array([t[0] for t in trades])
volumes = np.array([t[1] for t in trades])

# VWAP = sum(price * volume) / sum(volume)
vwap = np.sum(prices * volumes) / np.sum(volumes)
simple_avg = np.mean(prices)

print("=" * 55)
print("VWAP CALCULATION — BTC TRADE SERIES")
print("=" * 55)
print(f"{'Trade #':<10}{'Price':>12}{'Volume (BTC)':>14}{'Price × Vol':>16}")
print("-" * 55)
for i, (p, v) in enumerate(zip(prices, volumes)):
    print(f"{i+1:<10}${p:>11,.0f}{v:>14.1f}${p*v:>15,.0f}")
print("-" * 55)
print(f"{'Total':<10}{'':>12}{np.sum(volumes):>14.1f}${np.sum(prices*volumes):>15,.0f}")
print("=" * 55)
print(f"VWAP:                 ${vwap:>10,.2f}")
print(f"Simple Average Price: ${simple_avg:>10,.2f}")
print(f"Difference:           ${vwap - simple_avg:>+10,.2f}")
print("=" * 55)
print("\n💡 VWAP > simple avg because larger volume traded at higher prices.")

### Q2.4 — Sharpe Ratio Calculation

**Question:** A crypto trading strategy has the following monthly returns. Calculate the annualised Sharpe ratio. Assume a risk-free rate of 5% per annum.

**Risk-Free Rate Conversion**

$$ r_f^{\text{monthly}} = \frac{r_f^{\text{annual}}}{12} $$

---

**Monthly Returns**

$$ r_{\text{monthly}} = \frac{r_{\text{monthly (pct)}}}{100} $$

---

**Excess Returns**

$$ \text{Excess Return}_t = r_{\text{monthly}, t} - r_f^{\text{monthly}} $$

---

**Sharpe Ratio Components**

$$
\begin{align*}
\text{Mean Excess Return} &= \mu_{\text{excess}} = \mathbb{E}[r_{\text{excess}}] \\
\text{Std Dev of Excess Return} &= \sigma_{\text{excess}} = \sqrt{\text{Var}(r_{\text{excess}})}
\end{align*}
$$

---

**Monthly Sharpe Ratio**

$$ \text{Monthly Sharpe} = \frac{\mu_{\text{excess}}}{\sigma_{\text{excess}}} $$

---

**Annualised Sharpe Ratio**

$$ \text{Annualised Sharpe} = \text{Monthly Sharpe} \times \sqrt{12} $$

---

**Annualised Return (Geometric)**

$$ \text{Annualised Return} = \left( \prod_{i=1}^{12} (1 + r_{\text{monthly}, i}) \right) - 1 $$

---

**Annualised Volatility**

$$ \text{Annualised Volatility} = \sigma_{\text{excess}} \times \sqrt{12} \times 100\% $$

In [ ]:
# Q2.4 — Sharpe Ratio Calculation

# Monthly returns (%) for a BTC momentum strategy
monthly_returns_pct = np.array([
    8.2, -3.1, 12.5, 5.4, -7.8, 15.2,
    3.3, -1.2, 9.7, 6.1, -4.5, 11.3
])

annual_rfr = 0.05  # 5% p.a.
monthly_rfr = annual_rfr / 12

monthly_returns = monthly_returns_pct / 100

# Excess returns
excess_returns = monthly_returns - monthly_rfr

# Sharpe components
mean_excess   = np.mean(excess_returns)
std_excess    = np.std(excess_returns, ddof=1)  # sample std dev

# Monthly Sharpe → annualise by multiplying by sqrt(12)
monthly_sharpe    = mean_excess / std_excess
annualised_sharpe = monthly_sharpe * np.sqrt(12)

# Additional stats
annualised_return = (np.prod(1 + monthly_returns) ** 1) - 1  # 12 months here = 1 year
annualised_vol    = std_excess * np.sqrt(12) * 100

print("=" * 50)
print("SHARPE RATIO — CRYPTO STRATEGY")
print("=" * 50)
print(f"Monthly Returns: {monthly_returns_pct}%")
print("-" * 50)
print(f"Annual Risk-Free Rate:    {annual_rfr*100:.1f}%")
print(f"Monthly Risk-Free Rate:   {monthly_rfr*100:.4f}%")
print(f"Mean Monthly Excess Ret:  {mean_excess*100:.4f}%")
print(f"Std Dev (monthly):        {std_excess*100:.4f}%")
print("-" * 50)
print(f"Monthly Sharpe:           {monthly_sharpe:.4f}")
print(f"Annualised Sharpe:        {annualised_sharpe:.4f}  ← Key metric")
print(f"Annualised Return:        {annualised_return*100:.2f}%")
print(f"Annualised Volatility:    {annualised_vol:.2f}%")
print("=" * 50)
print()
if annualised_sharpe > 2:
    print("Rating: ⭐⭐⭐ Excellent (>2.0 — institutional quality)")
elif annualised_sharpe > 1:
    print("Rating: ⭐⭐ Good (1.0–2.0 — acceptable for crypto)")
elif annualised_sharpe > 0.5:
    print("Rating: ⭐ Below average (0.5–1.0 — needs improvement)")
else:
    print("Rating: ❌ Poor (<0.5)")

### Q2.5 — Kelly Criterion: Optimal Position Sizing

**Question:** Your crypto trading strategy has a 60% win rate. Average win is 2R, average loss is 1R. What fraction of capital should you risk per trade according to Kelly Criterion?

**Kelly Fraction**

$$ kelly\_fraction = \frac{b \cdot p - q}{b} $$

---

**Half Kelly (Common Conservative Approach)**

$$ half\_kelly = \frac{kelly\_fraction}{2} $$

---

**Why use Half Kelly?**
- Full Kelly is theoretically optimal for long-term growth but leads to very high volatility.
- Most professional traders use **Half Kelly** (or even 1/4 Kelly) for better risk-adjusted growth and psychological comfort.
- Reduces drawdowns significantly while still capturing most of the growth benefit.

In [ ]:
# Q2.5 — Kelly Criterion for Crypto Position Sizing

# Kelly Formula: f* = (b*p - q) / b
# where: b = net odds (avg win / avg loss), p = win prob, q = loss prob

p = 0.60    # win probability
q = 1 - p   # loss probability
b = 2.0     # win/loss ratio (avg win = 2x avg loss)

kelly_fraction = (b * p - q) / b
half_kelly = kelly_fraction / 2  # Traders often use 1/2 Kelly for safety

print("=" * 50)
print("KELLY CRITERION — POSITION SIZING")
print("=" * 50)
print(f"Win Probability (p):    {p:.0%}")
print(f"Loss Probability (q):   {q:.0%}")
print(f"Win/Loss Ratio (b):     {b:.1f}x")
print("-" * 50)
print(f"Full Kelly:             {kelly_fraction:.2%} of capital")
print(f"Half Kelly (safer):     {half_kelly:.2%} of capital")
print("=" * 50)
print()
print("Practical application (e.g. $1,000,000 fund):")
fund_size = 1_000_000
print(f"  Full Kelly bet size:  ${fund_size * kelly_fraction:>10,.0f}")
print(f"  Half Kelly bet size:  ${fund_size * half_kelly:>10,.0f}")
print()
print("💡 In crypto, practitioners use 1/4 to 1/2 Kelly due to:")
print("   - Fat-tail risk (crypto has extreme kurtosis)")
print("   - Parameter estimation uncertainty")
print("   - Correlated positions across crypto assets")

### Q2.6 — Cross-Exchange Arbitrage Profit

**Question:** BTC trades at $64,980 on Exchange A and $65,100 on Exchange B. Transaction costs are 0.05% per leg. Is this a profitable arb? What is the minimum spread needed to break even?

In [ ]:
# Q2.6 — Cross-Exchange Arbitrage (Statistical Arb)

price_A = 64_980   # Buy on Exchange A (cheaper)
price_B = 65_100   # Sell on Exchange B (higher)
fee_rate = 0.0005  # 0.05% per leg (taker fee)
btc_size = 1       # 1 BTC

# Costs
buy_fee  = price_A * fee_rate * btc_size
sell_fee = price_B * fee_rate * btc_size
total_cost = buy_fee + sell_fee

# Raw spread
gross_spread = price_B - price_A
net_profit   = gross_spread - total_cost
net_profit_pct = (net_profit / price_A) * 100

# Break-even spread
breakeven_spread = total_cost
breakeven_spread_pct = (breakeven_spread / price_A) * 100

print("=" * 55)
print("CROSS-EXCHANGE BTC ARBITRAGE")
print("=" * 55)
print(f"Buy  on Exchange A:   ${price_A:,.0f}")
print(f"Sell on Exchange B:   ${price_B:,.0f}")
print(f"Gross Spread:         ${gross_spread:,.0f}")
print("-" * 55)
print(f"Buy Fee (0.05%):      ${buy_fee:,.2f}")
print(f"Sell Fee (0.05%):     ${sell_fee:,.2f}")
print(f"Total Fees:           ${total_cost:,.2f}")
print("-" * 55)
print(f"Net Profit (1 BTC):   ${net_profit:,.2f}")
print(f"Net Profit %:         {net_profit_pct:.4f}%")
print("=" * 55)
print(f"Break-even spread:    ${breakeven_spread:,.2f} ({breakeven_spread_pct:.4f}%)")
print("=" * 55)
print()
if net_profit > 0:
    print(f"✅ PROFITABLE — Gross spread (${gross_spread}) > Fees (${total_cost:.2f})")
    print(f"   But consider: transfer time, slippage, capital lock-up!")
else:
    print(f"❌ NOT PROFITABLE after fees")

print()
print("⚠️  Real-world considerations:")
print("   1. Transfer time risk: price may converge before funds arrive")
print("   2. Pre-funded accounts on both exchanges needed for instant arb")
print("   3. Withdrawal fees, network gas costs (if on different chains)")
print("   4. Exchange risk / counterparty risk (post-FTX)")

### Q2.7 — Value at Risk (VaR) Calculation

**Question:** A crypto portfolio has an average daily return of 0.2% and daily standard deviation of 4.5%. Calculate the 1-day 95% and 99% VaR for a $10M portfolio.

### Value at Risk (VaR) Formulas

$$
\begin{align*}
\text{Parametric Daily VaR (95\%)} &= 1.645 \times \sigma_{\text{daily}} \times V \\[6pt]
\text{Parametric Daily VaR (99\%)} &= 2.326 \times \sigma_{\text{daily}} \times V \\[10pt]
\text{Historical VaR (95\%)} &= \text{Percentile}(r_{1..n}, 5\%) \times V \\[8pt]
\text{Expected Shortfall (CVaR)} &= \text{Mean of returns below VaR threshold} \times V
\end{align*}
$$

**Where:**
- \( V \) = Current Portfolio / Position Value
- \( \sigma_{\text{daily}} \) = Standard deviation of daily returns
- \( r \) = Historical returns series

In [ ]:
# Q2.7 — Parametric VaR for Crypto Portfolio

from scipy import stats

portfolio_value = 10_000_000  # $10M
daily_mean_ret  = 0.002       # 0.2% per day
daily_std       = 0.045       # 4.5% daily vol (crypto is volatile!)

confidence_levels = [0.90, 0.95, 0.99]

print("=" * 55)
print("PARAMETRIC VaR — CRYPTO PORTFOLIO")
print("=" * 55)
print(f"Portfolio Value:      ${portfolio_value:,.0f}")
print(f"Daily Mean Return:    {daily_mean_ret*100:.2f}%")
print(f"Daily Std Dev:        {daily_std*100:.2f}%")
print("-" * 55)
print(f"{'Confidence':<15}{'Z-Score':>10}{'VaR (%)':>12}{'VaR ($)':>16}")
print("-" * 55)

for cl in confidence_levels:
    z = stats.norm.ppf(1 - cl)        # negative z-score for loss
    var_pct = -(daily_mean_ret + z * daily_std)
    var_dollar = var_pct * portfolio_value
    print(f"{cl*100:.0f}%{' ':>10}{abs(z):>10.4f}{var_pct*100:>11.2f}%  ${var_dollar:>14,.0f}")

print("=" * 55)
print()
print("💡 Interpretation of 99% 1-day VaR:")
var_99 = (-(daily_mean_ret + stats.norm.ppf(0.01) * daily_std)) * portfolio_value
print(f"   On 99% of days, losses will NOT exceed ${var_99:,.0f}")
print(f"   On ~2-3 days per year, losses may exceed this threshold.")
print()
print("⚠️  VaR limitations in crypto:")
print("   - Returns are NOT normally distributed (fat tails, kurtosis ~15+)")
print("   - Better methods: Historical VaR, Monte Carlo, or Expected Shortfall (CVaR)")
print("   - Crypto can have 5-sigma events (BTC -40% in 24hrs, March 2020)")

### Q2.8 — Delta Hedging: Implied Volatility & Options Pricing

**Question:** BTC is at $65,000. A 1-week ATM call option has an implied volatility of 80% annualised. Using Black-Scholes, approximate the option's price and delta.

### Black-Scholes Formulas

**d1 and d2**

$$
\begin{align*}
d_1 &= \frac{\ln\left(\frac{S}{K}\right) + \left(r + \frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}} \\[8pt]
d_2 &= d_1 - \sigma \sqrt{T}
\end{align*}
$$

---

**Call Option Price**

$$
C = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)
$$

**Put Option Price**

$$
P = K e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)
$$

---

### Option Greeks

$$
\begin{align*}
\Delta_{\text{call}} &= N(d_1) \\
\Delta_{\text{put}} &= N(d_1) - 1 \\[8pt]
\Gamma &= \frac{N'(d_1)}{S \sigma \sqrt{T}} \\[8pt]
\Theta_{\text{call}} &= -\frac{S N'(d_1) \sigma}{2\sqrt{T}} - r K e^{-rT} N(d_2) \\[8pt]
\text{Vega} &= S \sqrt{T} N'(d_1)
\end{align*}
$$

**Where:**
- \( S \) = Spot price
- \( K \) = Strike price
- \( T \) = Time to expiration (in years)
- \( r \) = Risk-free interest rate
- \( \sigma \) = Volatility (annualized)
- \( N(\cdot) \) = Cumulative Normal Distribution
- \( N'(\cdot) \) = Normal Probability Density Function

In [ ]:
# Q2.8 — Black-Scholes Option Pricing (Crypto)

from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    """Black-Scholes option pricing formula."""
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
    else:
        price = K * math.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1

    gamma  = norm.pdf(d1) / (S * sigma * math.sqrt(T))
    theta  = (-(S * norm.pdf(d1) * sigma) / (2 * math.sqrt(T)) - r * K * math.exp(-r * T) * norm.cdf(d2)) / 365
    vega   = S * norm.pdf(d1) * math.sqrt(T) / 100  # per 1% vol change

    return price, delta, gamma, theta, vega, d1, d2

# BTC Option Parameters
S     = 65_000   # Spot price
K     = 65_000   # Strike (ATM)
T     = 7/365    # 1 week to expiry (in years)
r     = 0.05     # 5% risk-free rate
sigma = 0.80     # 80% IV (annualised)

price, delta, gamma, theta, vega, d1, d2 = black_scholes(S, K, T, r, sigma)

print("=" * 55)
print("BLACK-SCHOLES — BTC CALL OPTION")
print("=" * 55)
print(f"Spot Price (S):        ${S:,.0f}")
print(f"Strike Price (K):      ${K:,.0f}  [ATM]")
print(f"Time to Expiry (T):    {T*365:.0f} days ({T:.4f} years)")
print(f"Implied Vol (σ):       {sigma*100:.0f}% annualised")
print(f"Risk-Free Rate (r):    {r*100:.1f}%")
print("-" * 55)
print(f"d1:                    {d1:.4f}")
print(f"d2:                    {d2:.4f}")
print("=" * 55)
print(f"OPTION PRICE:          ${price:>10,.2f} per BTC")
print(f"  As % of spot:        {price/S*100:.2f}%")
print("=" * 55)
print("GREEKS:")
print(f"  Delta (Δ):           {delta:.4f}  [0.5 = ATM]")
print(f"  Gamma (Γ):           {gamma:.6f}")
print(f"  Theta (Θ):           ${theta:,.2f}/day  [time decay]")
print(f"  Vega (ν):            ${vega:,.2f} per 1% IV move")
print("=" * 55)
print()
print("💡 Delta Hedging:")
print(f"   To be delta-neutral when SHORT this call:")
print(f"   BUY {delta:.4f} BTC per option contract")
print(f"   = BUY ${delta * S:,.0f} worth of BTC")

---
<a id='microstructure'></a>
## 3. Market Microstructure & Order Types


### Q3.1 — What is a Central Limit Order Book (CLOB) and how does it work?

**Answer:**  
A **CLOB** matches buy and sell orders by price-time priority:
1. **Price priority:** Best price gets matched first (highest bid, lowest ask)
2. **Time priority:** If two orders have the same price, earlier order fills first

**Order book example (BTC/USDT):**  
```
ASK side (sellers):
  $65,100 — 2.5 BTC
  $65,050 — 1.2 BTC   ← Best ask
  
[MID: $65,025]

BID side (buyers):
  $65,000 — 3.0 BTC   ← Best bid
  $64,950 — 5.0 BTC
```
Spread = $65,050 − $65,000 = $50

**Matching engine:** If a market buy order arrives for 2 BTC, it fills against the best ask ($65,050) — 1.2 BTC fills, then 0.8 BTC fills at $65,100 (partial).

### Q3.2 — Explain all major order types used in crypto trading.

**Answer:**  

| Order Type | Description | Use Case |
|---|---|---|
| **Market Order** | Execute immediately at best available price | Urgent fills; accepts slippage |
| **Limit Order** | Execute only at specified price or better | Precise entry/exit; no slippage but may not fill |
| **Stop Market** | Market order triggered when price hits stop level | Stop-loss protection |
| **Stop Limit** | Limit order triggered when stop hit | More control but may not fill in fast markets |
| **IOC (Immediate or Cancel)** | Fill immediately, cancel remainder | Large orders; avoid partial hangs |
| **FOK (Fill or Kill)** | Fill entire order or cancel completely | All-or-nothing execution |
| **Post-Only** | Only execute as maker (never taker) | Rebate capture; market making |
| **Iceberg/Hidden** | Show only portion of total order | Large orders to avoid signalling |
| **TWAP** | Split order over time intervals | Algorithmic execution |
| **Trailing Stop** | Stop moves with price by fixed % | Lock in profits on trending crypto |

**Crypto-specific:** On DEXs, orders are **transactions** — they are broadcast to a mempool and can be **front-run by MEV bots** (Miner Extractable Value). Slippage tolerance settings on DEXs act as a form of stop-limit protection.

### Q3.3 — What is latency arbitrage and how do crypto HFT firms exploit it?

**Answer:**  
**Latency arbitrage** exploits the time delay between when a price move occurs on one venue and when it is reflected on another.

**Crypto example:**  
1. BTC drops $200 on Coinbase due to a large sell order at 14:00:00.001  
2. Other exchanges (Kraken, Gemini) still show the old price for ~50ms  
3. An HFT firm co-located near all exchanges sees the Coinbase print first  
4. In microseconds, they sell on Kraken/Gemini before prices update  

**Infrastructure required:**  
- Co-location at exchange data centres  
- Custom FPGA / kernel-bypass networking  
- Direct market data feeds (not aggregated APIs)  
- Sub-millisecond execution systems  

**Note:** In crypto, this is more accessible than equities HFT since exchange co-location is cheaper and WebSocket APIs are widely available.

---
<a id='crypto'></a>
## 4. Crypto-Specific Concepts

### Q4.1 — What is MEV (Miner/Maximum Extractable Value) and how does it affect trading?

**Answer:**  
**MEV** is profit that can be extracted from blockchain transaction ordering by block producers (miners/validators).

**Types of MEV:**  
- **Front-running:** See a pending large swap in mempool → insert your tx before it at higher gas → profit from the price impact  
- **Back-running:** Insert tx immediately after a known large tx to capture price recovery  
- **Sandwich attack:** Front-run + back-run the same victim tx  
- **Liquidation MEV:** Race to be first to liquidate underwater DeFi positions (Aave, Compound)  
- **Arbitrage MEV:** Exploit price differences across DEX pools atomically  

**Scale:** Ethereum alone has seen $700M+ in extracted MEV. MEV bots pay priority gas fees (tips) competing for this value.

**Trading implication:** Large DEX trades use **slippage protection**, **private mempools** (Flashbots Protect), or **MEV-resistant DEXs** (CoW Protocol) to avoid being sandwiched.

### Q4.2 — Explain the mechanics of a DeFi liquidation and how a trading firm profits from it.

**Answer:**  
**DeFi Lending example (Aave/Compound):**  
1. User deposits 10 ETH as collateral, borrows $20,000 USDC  
2. ETH price drops: collateral value falls below the **liquidation threshold** (e.g. Health Factor < 1.0)  
3. Anyone can call the `liquidate()` function  
4. Liquidator repays part of the debt → receives the collateral at a **5-15% discount**  

**Trading firm strategy:**  
- Monitor all positions across Aave/Compound/MakerDAO on-chain  
- Pre-compute liquidation prices for every position  
- Use **flash loans** — borrow capital instantly (same tx) to liquidate without upfront capital  
- Immediately sell received collateral to lock in the discount  

**Key metric:** Liquidation bonus (usually 5–15%) minus gas fees and slippage = net profit per liquidation

### Q4.3 — What is an Automated Market Maker (AMM) and how does impermanent loss work?

**Answer:**  
An **AMM** uses a mathematical formula instead of an order book to price assets.

**Constant Product formula (Uniswap v2):** `x × y = k`  
Where x = token A quantity, y = token B quantity, k = constant

**Impermanent Loss (IL):** The loss suffered by a liquidity provider when the price of deposited assets changes vs simply holding them.

**Formula:** `IL = 2√(price_ratio) / (1 + price_ratio) − 1`

In [ ]:
# Q4.3 — Impermanent Loss Calculation

def impermanent_loss(price_ratio):
    """Calculate IL for a Uniswap-style AMM pool.
    price_ratio = new_price / initial_price
    """
    il = 2 * math.sqrt(price_ratio) / (1 + price_ratio) - 1
    return il

print("=" * 50)
print("IMPERMANENT LOSS — BTC/USDC POOL")
print("=" * 50)
print(f"{'Price Change':>15}{'Price Ratio':>14}{'IL':>10}")
print("-" * 50)

price_changes = [-75, -50, -25, 0, 25, 50, 100, 200, 400]

for pct in price_changes:
    ratio = 1 + pct/100
    if ratio <= 0:
        continue
    il = impermanent_loss(ratio)
    print(f"{pct:>+14}%{ratio:>13.2f}x{il*100:>9.2f}%")

print("=" * 50)
print()
print("Example: Deposit $10,000 worth of BTC+USDC into pool")
print("BTC price doubles (100% increase):")
il_example = impermanent_loss(2.0)
print(f"  IL = {il_example*100:.2f}%")
print(f"  Value in pool:  ${10000 * (1 + il_example):,.2f}")
print(f"  Value if HODL:  ${10000 * 1.5:,.2f}  (avg of 2x BTC + 1x USDC)")
print(f"  Loss from LP:   ${10000 * 1.5 - 10000 * (1 + il_example):,.2f}")
print()
print("💡 IL is 'impermanent' because if price reverts, loss disappears.")
print("   It becomes permanent when you withdraw from the pool.")

### Q4.4 — What happened with FTX in November 2022 and what risk management lessons apply?

**Answer:**  
**Timeline:**  
- Nov 2, 2022: CoinDesk published leaked Alameda Research balance sheet — heavily concentrated in FTT (FTX's native token)  
- Nov 6: Binance CEO announced selling all FTT holdings (worth ~$500M)  
- Nov 8: FTX suffered $6B in withdrawals in 72 hours — classic bank run  
- Nov 11: FTX filed for bankruptcy  

**Root causes:**  
1. **Co-mingling of client funds** — FTX lent customer deposits to Alameda  
2. **Illiquid collateral** — Balance sheet dominated by own-issued token (FTT)  
3. **Lack of risk controls** — No independent risk team, CEO bypassed compliance  
4. **Counterparty concentration** — Firms with all assets on FTX lost everything  

**Lessons for trading firms:**  
- Never keep more than 1-2 days' working capital on any single exchange  
- Demand **proof of reserves** (Merkle tree-based) from exchanges  
- Diversify exchange relationships — Binance, Kraken, Coinbase, OKX  
- Use self-custody cold storage for long-term holdings  
- Monitor counterparty concentration risk daily

---
<a id='risk'></a>
## 5. Risk Management Questions

### Q5.1 — What is Greeks exposure management in crypto options trading?

**Answer:**  

| Greek | Measures | Crypto context |
|---|---|---|
| **Delta (Δ)** | Price sensitivity | A delta-1 BTC book = same risk as holding 1 BTC |
| **Gamma (Γ)** | Rate of change of delta | High gamma near expiry/ATM — expensive to delta-hedge in volatile crypto |
| **Theta (Θ)** | Time decay | Options sellers earn theta; on Deribit this is very significant in 0DTE BTC options |
| **Vega (ν)** | IV sensitivity | Crypto vol can double overnight; long vega protects against vol spikes |
| **Rho (ρ)** | Interest rate sensitivity | Less relevant in crypto until stablecoin yield markets matured |

**Practical example:** A crypto options market maker might be delta-neutral and gamma/vega long — they earn by rebalancing as price moves while being protected from vol spikes.

### Q5.2 — How do you size a stop-loss in crypto given its extreme volatility?

**Answer:**  
**ATR-based stop loss** is preferred in crypto:

`Stop distance = N × ATR(14)`  
Where N = risk multiplier (typically 1.5x to 3x), ATR = Average True Range over 14 periods

**Position sizing formula:**  
`Position size = (Account risk $) / (Stop distance in $)`  
`Account risk = Account value × risk per trade %`

**Example:** BTC account $500k, risk 1% per trade, BTC ATR(14) = $1,500, stop at 2x ATR:
- Account risk = $500k × 1% = $5,000  
- Stop distance = 2 × $1,500 = $3,000  
- Position size = $5,000 / $3,000 = 1.67 BTC

**Key point:** Crypto volatility means stops need to be wider — tight stops get hunted in thin markets.

### Q5.3 — What is counterparty risk and how is it managed in crypto?

**Answer:**  
**Counterparty risk** = the risk that the other party in a trade or custody arrangement defaults.

**In crypto, this includes:**  
1. **Exchange risk:** FTX, Mt. Gox, Celsius, BlockFi — all collapsed  
2. **Custodian risk:** Keeping assets with a third-party custodian  
3. **DeFi smart contract risk:** Protocol exploits (Wormhole: $320M, Ronin: $625M)  
4. **OTC counterparty risk:** Trading with a dealer who doesn't deliver  
5. **Stablecoin issuer risk:** USDC depeg during SVB collapse (Mar 2023)  

**Mitigation:**  
- Exchange limits: no more than 2-5% of AUM on any single exchange  
- Use regulated custodians (Coinbase Custody, Fidelity Digital)  
- DVP (Delivery vs Payment) settlement for OTC  
- Multi-sig wallets for self-custody  
- Diversify stablecoin exposure (USDC, USDT, PYUSD)

---
<a id='scenario'></a>
## 6. Scenario-Based Questions

### Q6.1 — Scenario: Flash Crash

**Question:** It's 3am UTC. ETH suddenly drops 30% in 90 seconds on Binance due to a large algorithmic sell order. Your firm has $10M long ETH. Walk me through your response.

**Strong Answer:**  

**Immediate (0-60 seconds):**  
1. **Verify legitimacy** — Is this an exchange glitch or real price movement? Check Coinbase, Kraken, Bybit simultaneously  
2. **Halt automated strategies** — Kill switches on all ETH algo strategies  
3. **Assess liquidation risk** — Are any leveraged positions approaching margin call levels?  

**Short-term (1-5 minutes):**  
4. **Prioritise survival** — Reduce or hedge the position if it's a real move  
5. **Check if this is CEX-isolated** — If only on Binance, the Binance/Coinbase basis trade is an opportunity  
6. **Look for liquidity** — Thin markets; any hedge execution will have massive slippage  

**Analysis:**  
7. **Root cause** — Large liquidation cascade? Exchange error? Spot only or also perps?  
8. **Post-trade review** — PnL attribution, risk limit review, incident report  

**Key interview signals to demonstrate:**  
- Composure under pressure  
- Process-oriented thinking (kill switches, escalation path)  
- Understanding of cross-venue dynamics  
- Risk management priority before opportunity-seeking

### Q6.2 — Scenario: Stablecoin Depegging

**Question:** USDT (Tether) starts trading at $0.95 on major exchanges. Your firm has $50M in USDT as operating capital. What do you do?

**Strong Answer:**  

1. **Immediate swap:** Convert USDT to USDC, BTC or ETH — accept the $2.5M haircut vs risking total loss  
2. **Assess severity:** Is this a temporary liquidity panic (like USDC/SVB) or a genuine insolvency event (like UST/Luna)?  
   - UST collapsed from $1 → $0 in 48 hours  
   - USDC temporarily depegged to $0.87 in March 2023 but recovered within 24 hours  
3. **Check on-chain:** Tether attestation reports, reserve composition, withdrawal queues  
4. **Hedge remaining exposure:** Buy CDS on Tether (if available) or maintain short via stablecoin basis  
5. **Diversify stablecoin holdings** going forward: 40% USDC / 40% USDT / 20% T-bills via BUIDL or ONDO  

**Key risk principle:** Stablecoins are NOT risk-free assets. They carry issuer risk, regulatory risk, and reserve quality risk.

### Q6.3 — Scenario: Market Making Inventory Risk

**Question:** Your firm is market making BTC/USDT. Over the past hour, you've accumulated a 50 BTC long inventory as you've been providing bids and the market has been selling. BTC is at $64,000. What do you do?

**Strong Answer:**  

**Analyse the situation:**  
- 50 BTC long = $3.2M directional exposure  
- This is an **inventory imbalance** — you've been consistently hit on bids = informed selling  
- Risk: BTC continues to fall, increasing losses  

**Options:**  
1. **Skew quotes:** Move both bid and ask down by ~$50-100 to attract buyers and discourage sellers. This helps offload inventory naturally without market impact.  
2. **Widen spreads:** Increase the spread to reduce adverse selection risk while you work off inventory  
3. **Hedge with perps:** Sell 50 BTC perpetual futures to delta-hedge the inventory — maintain market making activity but neutralise directional risk  
4. **Lift offers:** Actively cross the spread to sell 50 BTC into the market (costly but immediate)  
5. **Assess informed flow:** Is this institutional distribution? Check order flow metrics (trade size, aggressor ratios)  

**Best answer:** Combination of perp hedge (immediate) + quote skewing (gradual) + spread widening (self-protection).

### Q6.4 — Scenario: Regulatory Shock

**Question:** The SEC announces emergency action against 3 major crypto exchanges. Crypto markets drop 20% in 2 hours. How does this affect your buy-side crypto fund's strategy?

**Strong Answer:**  

**Immediate concerns:**  
1. **Counterparty exposure:** Are any of the 3 exchanges where our assets are held?  
2. **Liquidity:** Can we actually get our assets off those exchanges?  
3. **Hedging:** Deploy pre-planned risk reduction — sell futures, buy puts  

**Strategic assessment:**  
- **Permanent impairment or overreaction?** Evaluate legal merit of the SEC case  
- **Precedent:** Similar events (Binance/CFTC May 2023, Coinbase/SEC) saw 15-20% drops then partial recovery  
- **Regulatory arbitrage:** Capital may flow to offshore exchanges or decentralised protocols  

**Opportunity framing:**  
- If markets are pricing in permanent ban (tail risk), but reality is a fine/settlement — asymmetric long opportunity  
- Watch on-chain flows: Are whales accumulating or distributing?

**Communication:**  
- Proactively brief LPs on the situation, explain portfolio positioning, not just P&L

---
<a id='derivatives'></a>
## 7. Derivatives & Options

### Q7.1 — What is a volatility smile/skew in crypto options, and what does it tell you?

**Answer:**  
**Volatility smile:** When IV is plotted against strike prices, it forms a U-shape (or skewed shape) rather than being flat.

**In crypto specifically:**  
- BTC options typically exhibit **positive skew** — OTM calls have higher IV than OTM puts  
- This reflects crypto's **upside tail risk** — markets price in the possibility of explosive rallies (e.g., ETF approval, halving)  
- In traditional equities, it's the opposite (negative skew) due to crash fear  

**Practical use:**  
- High call skew = market paying premium for upside exposure → possible crowded long positioning  
- Selling OTM calls when skew is rich is a yield-enhancement strategy  
- Put-call skew ratio is a sentiment indicator used by trading desks daily

### Q7.2 — Explain a perpetual futures funding rate arbitrage trade in detail.

**Answer:**  
**Setup:** BTC perpetual funding = +0.05% every 8 hours (annualised: 0.05% × 3 × 365 = 54.75% p.a.)

**Trade:**  
1. **Buy 1 BTC spot** on Coinbase at $65,000  
2. **Short 1 BTC perpetual** on Bybit at ~$65,000  
3. Net delta = 0 (market neutral)  
4. Every 8 hours: collect funding payment as the short holder  

**Return calculation:**

In [ ]:
# Q7.2 — Funding Rate Arbitrage Return

spot_price      = 65_000
funding_rate_8h = 0.0005  # 0.05% per 8hr period
periods_per_day = 3
days            = 30

# Costs
spot_fee   = spot_price * 0.001  # 0.1% buy fee on spot
perp_fee   = spot_price * 0.0005 # 0.05% on perp open
total_cost = spot_fee + perp_fee

# Funding income
daily_funding = spot_price * funding_rate_8h * periods_per_day
total_funding = daily_funding * days

# Net profit
net_profit = total_funding - total_cost
annualised_yield = (total_funding / (days/365) / spot_price) * 100

print("=" * 55)
print("FUNDING RATE ARB — 30-DAY SIMULATION")
print("=" * 55)
print(f"Capital deployed:     ${spot_price:,.0f}")
print(f"Funding rate:         {funding_rate_8h*100:.3f}% per 8hr")
print("-" * 55)
print(f"Entry costs (fees):   ${total_cost:,.2f}")
print(f"Daily funding income: ${daily_funding:,.2f}")
print(f"30-day funding total: ${total_funding:,.2f}")
print(f"Net profit (30d):     ${net_profit:,.2f}")
print(f"Annualised yield:     {annualised_yield:.2f}%")
print("=" * 55)
print()
print("Risks:")
print("  1. Funding can flip negative (you start paying)")
print("  2. Spot vs perp basis divergence")
print("  3. Exchange risk on the perp leg")
print("  4. Margin call risk if perp moves adversely before funding")

---
<a id='defi'></a>
## 8. DeFi & On-Chain Trading

### Q8.1 — What is a flash loan and give an example of how a trading firm could use it?

**Answer:**  
A **flash loan** is an uncollateralised loan that must be borrowed and repaid within the same blockchain transaction. If repayment fails, the entire transaction reverts.

**Legitimate uses:**  
1. **Arbitrage:** Borrow $10M USDC, buy ETH on Uniswap at lower price, sell on Curve at higher price, repay loan + fee, keep spread — all in one atomic tx  
2. **Liquidations:** Borrow to liquidate an undercollateralised position, receive bonus collateral, repay loan  
3. **Collateral swap:** Replace risky collateral with safer collateral in one tx without prior capital  

**Fee:** Aave charges 0.09% for flash loans. Profit must exceed fee + gas.

**Risk:** Smart contract bugs in flash loan logic can be catastrophic. Always audit flash loan code.

### Q8.2 — Compare Uniswap v2 vs v3 from a liquidity provision standpoint.

**Answer:**  

| Feature | Uniswap v2 | Uniswap v3 |
|---|---|---|
| **Liquidity distribution** | Uniform (0 to ∞) | Concentrated within custom price ranges |
| **Capital efficiency** | Low — most liquidity unused | Up to 4000x more efficient |
| **IL risk** | Lower (spread wide) | Higher (concentrated range) |
| **Management** | Passive — set and forget | Active — range must be managed |
| **Fee tiers** | 0.30% fixed | 0.01% / 0.05% / 0.30% / 1.00% |
| **LP NFT** | ERC-20 fungible | ERC-721 NFT (unique position) |

**Trading firm implication:**  
- v3 is like market making — you set a bid/ask range and earn fees only while price is in range  
- Sophisticated funds deploy active v3 LP management strategies with delta-hedging — essentially DeFi market making  
- Key risk: price exits your range → you hold 100% of one asset (full conversion to the underperforming side)

---
<a id='behavioral'></a>
## 9. Behavioral / Fit Questions

### Q9.1 — Walk me through a trade you're proud of (or a trade that went wrong).

**Strong framework:**  
1. **Context:** What was the market environment? (Bull/bear, vol regime)  
2. **Thesis:** What was your edge or reasoning?  
3. **Execution:** How did you size it, when did you enter/exit?  
4. **Outcome:** What happened? What did you learn?  
5. **Reflection:** What would you do differently?

**Example narrative:** _"In Q4 2022, post-FTX collapse, crypto sentiment was at multi-year lows. I observed that BTC's realised volatility was compressing while the funding rate turned deeply negative. I saw this as a potential spring-loaded setup: extreme fear, compressed vol, and negative sentiment all historically preceded sharp recoveries. I built a modest long through Jan 2023 BTC calls on Deribit with defined max loss. BTC rallied from $15k to $23k in Q1 2023. I took profit at 200% on the options. Lesson: extreme sentiment extremes in crypto often precede sharp reversals, but position sizing discipline is critical."_

### Q9.2 — How do you stay informed about crypto markets?

**Strong Answer (structured):**  

**On-chain / data:**  
- Glassnode, Nansen — on-chain flows, whale wallets, exchange reserves  
- CryptoQuant — exchange inflows, miner behaviour  
- Dune Analytics — custom on-chain queries  

**Market structure:**  
- Coinalyze, Velo Data — open interest, funding rates, liquidation maps  
- Laevitas — options vol surface, skew  

**News / research:**  
- The Block, CoinDesk, Blockworks  
- Twitter/X: follow key developers, protocol teams, traders  
- Messari, Delphi Digital for in-depth research  

**Trading community:**  
- Telegram/Discord groups for specific protocols  
- Read governance proposals for major DeFi protocols

### Q9.3 — Sell-side specific: How do you handle a client who wants to buy $200M of BTC?

**Strong Answer:**  

1. **Assess market impact:** $200M BTC ≈ ~3,000 BTC. Daily BTC spot volume is $15-25B — this is 1-2% of daily volume. Significant but executable.  
2. **Quote a spread/all-in price:** Based on current market depth and anticipated slippage. Typically 20-40bps for a block this size.  
3. **Execution strategy:**  
   - TWAP over 4-8 hours across Binance, Coinbase, OKX  
   - Use dark pool / block trading venues (Paradigm OTC)  
   - Consider futures to pre-hedge while aggregating spot  
4. **Risk management:** If principal trade, hedge the residual immediately. Define max loss tolerance.  
5. **Client communication:** Provide real-time trade reports, confirm settlement timing, handle compliance (source of funds, KYC).

### Q9.4 — How do you think about the difference between buy-side and sell-side roles in crypto?

**Answer:**  

| Dimension | Buy-Side | Sell-Side |
|---|---|---|
| **Goal** | Generate alpha for the fund | Provide services (execution, liquidity, research) to clients |
| **Revenue** | Performance fees + management fees | Spread, commissions, advisory fees |
| **P&L ownership** | Direct — your positions = your P&L | Indirect — client flow P&L + proprietary book |
| **Examples** | Pantera, a16z crypto, Multicoin, crypto hedge funds | Coinbase Institutional, Galaxy, FalconX, Wintermute |
| **Day-to-day** | Research, portfolio management, risk monitoring | Market making, OTC trading, client service, structuring |
| **Edge required** | Informational, analytical, structural | Speed, client relationships, risk management |

**Why crypto is unique:** The line is blurrier — many sell-side firms (market makers) also run proprietary books, and buy-side firms may provide liquidity in DeFi to earn yield.

---
## 📝 Quick Reference Cheat Sheet

| Formula | Equation |
|---|---|
| **Bid-Ask Spread** | Ask − Bid |
| **Slippage %** | (Fill − Expected) / Expected × 100 |
| **VWAP** | Σ(P × V) / ΣV |
| **Funding PnL** | Position Size × Rate × Periods |
| **Sharpe (annualised)** | (Mean Excess Return / Std Dev) × √252 (daily) or √12 (monthly) |
| **Kelly Fraction** | (b×p − q) / b |
| **Parametric VaR** | μ − z×σ (as % of portfolio) |
| **Impermanent Loss** | 2√r/(1+r) − 1 where r = price ratio |
| **Liquidation Price (long)** | Entry × (1 − 1/leverage + maint. margin rate) |
| **Basis** | Futures Price − Spot Price |
| **Annualised Funding** | Rate per period × periods per day × 365 |

---
*This notebook covers key interview topics for crypto buy/sell-side trading roles. Good luck! 🚀*